# EMG → Multi‑Label Fingers with a Spiking Neural Network (Temporal Dependence)

This notebook reimplements your pipeline using **PyTorch + snnTorch**.

In [1]:
# %% [markdown]
# ## 0) Imports & Config

import os, re, json, math, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import snntorch as snn
from snntorch import surrogate

from sklearn.metrics import r2_score, roc_auc_score

from exg.ema import EMA
from exg.sma import SMA
from exg.iir import IIR

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
print("Device:", DEVICE)

torch.manual_seed(42)
np.random.default_rng(42)

Device: mps


Generator(PCG64) at 0x33B0984A0

In [2]:
# %% [markdown]
# ## 1) Load + Preprocess

def load_data(session_dir: str,
              finger_columns: List[str],
              window_sizes: List[int],
              exg_fs: int):
    exg_path    = os.path.join(session_dir, "exg.csv")
    prompt_path = os.path.join(session_dir, "prompt.csv")
    exg_data    = pd.read_csv(exg_path)
    angle_data  = pd.read_csv(prompt_path)

    raw_columns   = [c for c in exg_data.columns if c != "timestamp"]
    angle_columns = [c for c in finger_columns if c in angle_data.columns]

    exg_data['timestamp']   = pd.to_timedelta(exg_data['timestamp'], unit='ms')
    angle_data['timestamp'] = pd.to_timedelta(angle_data['timestamp'], unit='ms')

    iir_params = {"num_channels": len(raw_columns), "fs": exg_fs,
                  "lowpass_fs": exg_fs/2, "highpass_fs": 20,
                  "notch_fs_list": [50, 60], "filter_order": 4}
    iir_proc = IIR(**iir_params)
    exg_data[raw_columns] = iir_proc.process(exg_data[raw_columns].values)

    ema_proc = EMA(window_sizes, len(raw_columns), fs=exg_fs)
    sma_proc = SMA(window_sizes, len(raw_columns), fs=exg_fs)
    ema_df   = ema_proc.results_to_df(ema_proc.process(exg_data[raw_columns].values))
    sma_df   = sma_proc.results_to_df(sma_proc.process(exg_data[raw_columns].values))

    processed_exg = pd.concat([exg_data, ema_df, sma_df], axis=1)
    merged = pd.merge_asof(processed_exg.sort_values('timestamp'),
                           angle_data[angle_columns+["timestamp"]].sort_values('timestamp'),
                           on='timestamp', direction='backward',
                           tolerance=pd.Timedelta(milliseconds=5000))
    return merged.dropna(), raw_columns, list(ema_df.columns), list(sma_df.columns), iir_params


def load_session(session_folder: str,
                 finger_columns: List[str],
                 window_sizes: List[int]):
    with open(os.path.join(session_folder, "frequency_info.json")) as f:
        exg_fs = json.load(f)["exg_fs"]

    dirs = [d for d in os.listdir(session_folder)
            if os.path.isdir(os.path.join(session_folder,d)) and re.match(r"^r_(\d+)$", d)]
    dirs.sort(key=lambda d: int(d.split('_')[1]))

    sessions=[]; ema_cols=sma_cols=iir_params=None
    for d in dirs:
        df, raw, ema_cols, sma_cols, iir_params = load_data(
            os.path.join(session_folder,d), finger_columns, window_sizes, exg_fs)
        sessions.append(df)
    return sessions, exg_fs, raw, ema_cols, sma_cols, iir_params

In [3]:
# %% [markdown]
# ## 2) Parameters + Run Loader

window_sizes   = [128]
finger_columns = ["thumb","index","middle","ring","pinky"]
session_folder = "../data/s_04_26_25"

sessions, exg_fs, exg_cols, ema_cols, sma_cols, iir_params = load_session(
    session_folder, finger_columns, window_sizes)

for i, df in enumerate(sessions):
    print(f"Session {i}: {df.shape}")

TypeError: EMA.__init__() got multiple values for argument 'fs'

In [4]:
# %% [markdown]
# ## 3) Train/Val Split + Normalization
def drop_nan_sessions(slist): return [d.dropna() for d in slist]

preprocessed = drop_nan_sessions(sessions)
x_cols = ema_cols[:15]
val_idx=[len(preprocessed)-1]; train_idx=list(set(range(len(preprocessed)))-set(val_idx))

X_train,y_train,X_val,y_val=[],[],[],[]
delay=4
for s in train_idx:
    X=preprocessed[s][x_cols].to_numpy(); Y=preprocessed[s][finger_columns].to_numpy()
    X_train.append(X[:-delay]); y_train.append(Y[delay:])
for s in val_idx:
    X=preprocessed[s][x_cols].to_numpy(); Y=preprocessed[s][finger_columns].to_numpy()
    X_val.append(X[:-delay]); y_val.append(Y[delay:])
X_train=np.concatenate(X_train); y_train=np.concatenate(y_train)
X_val=np.concatenate(X_val); y_val=np.concatenate(y_val)

train_mean=X_train.mean(0); train_std=X_train.std(0)+1e-8
X_train=(X_train-train_mean)/train_std; X_val=(X_val-train_mean)/train_std
print(X_train.shape, y_train.shape)

NameError: name 'sessions' is not defined

In [5]:
# %% [markdown]
# ## 4) Sequence Dataset
class SeqDataset(Dataset):
    def __init__(self,X,Y,T):
        self.X=X.astype(np.float32); self.Y=Y.astype(np.float32); self.T=T; self.N=len(X)-T+1
        if self.N < 1:
            raise ValueError("Sequence length longer than stream")
    def __len__(self): return self.N
    def __getitem__(self,idx):
        return torch.from_numpy(self.X[idx:idx+self.T]), torch.from_numpy(self.Y[idx:idx+self.T])

SEQ_LEN=50; BATCH=256
train_dl=DataLoader(SeqDataset(X_train,y_train,SEQ_LEN), batch_size=BATCH,shuffle=True,drop_last=True)
val_dl=DataLoader(SeqDataset(X_val,y_val,SEQ_LEN), batch_size=BATCH)
in_dim, out_dim=X_train.shape[1], y_train.shape[1]
print("in_dim, out_dim:", in_dim, out_dim)

NameError: name 'X_train' is not defined

In [6]:
# %% [markdown]
# ## 5) Spiking Model
class SpikingTemporalNet(nn.Module):
    def __init__(self,in_dim,hid,out_dim):
        super().__init__()
        self.fc=nn.Linear(in_dim,hid)
        self.rlif=snn.RLeaky(beta=0.9,threshold=1.0,surrogate_function=surrogate.fast_sigmoid())
        self.fc_out=nn.Linear(hid,out_dim)
    def forward(self,x):
        B,T,C=x.shape
        mem=h=torch.zeros(B,self.fc.out_features,device=x.device)
        preds=[]
        for t in range(T):
            cur=self.fc(x[:,t,:])
            spk,mem,h=self.rlif(cur,mem,h)
            preds.append(self.fc_out(spk))
        return torch.stack(preds,1)

model=SpikingTemporalNet(in_dim,128,out_dim).to(DEVICE)
criterion=nn.BCEWithLogitsLoss(); opt=torch.optim.Adam(model.parameters(),1e-3, weight_decay=1e-5)

NameError: name 'in_dim' is not defined

In [7]:
# %% [markdown]
# ## 6) Training & Eval
@torch.no_grad()
def eval_epoch(model,loader):
    model.eval(); logits=[]; ys=[]
    for xb,yb in loader:
        l=model(xb.to(DEVICE)).cpu(); logits.append(l); ys.append(yb)
    L=torch.cat(logits).numpy(); Y=torch.cat(ys).numpy()
    P=1/(1+np.exp(-L));
    flatY=Y.reshape(-1, Y.shape[-1]); flatP=P.reshape(-1, P.shape[-1])
    r2=[max(r2_score(flatY[:,i],flatP[:,i]),0) for i in range(flatY.shape[1])]
    return np.mean(r2)

def train(model,train_dl,val_dl,epochs=50,patience=8):
    best=-1e9; wait=0
    for e in range(1,epochs+1):
        model.train(); tot=0; n=0
        for xb,yb in train_dl:
            xb,yb=xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True); logits=model(xb); loss=criterion(logits,yb)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
            tot+=loss.item(); n+=1
        val_r2=eval_epoch(model,val_dl)
        print(f"Epoch {e:03d} | loss={tot/max(n,1):.4f} | val R2={val_r2:.4f}")
        if val_r2 > best + 1e-4:
            best=val_r2; best_state={k:v.cpu() for k,v in model.state_dict().items()}; wait=0
        else:
            wait+=1
            if wait>=patience:
                print("Early stopping."); break
    model.load_state_dict(best_state)

train(model,train_dl,val_dl,epochs=100,patience=10)

NameError: name 'model' is not defined

In [8]:
# %% [markdown]
# ## 7) Save Model + Params
def extract_method_interval_xcols(x_cols: list, xma: str="ema"):
    methods, intervals=set(), set()
    for col in x_cols:
        if xma in col:
            split=col.split('_')
            method="_".join(split[3:-1])
            try: interval=int(split[-1][:-2])
            except: continue
            methods.add(method); intervals.add(interval)
    return sorted(list(methods)), sorted(list(intervals))

def save_model(model_dir="../data/s_04_26_25/models/snn"):
    os.makedirs(model_dir, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(model_dir, "model.pt"))
    ema_methods, ema_intervals = extract_method_interval_xcols(x_cols, "ema")
    sma_methods, sma_intervals = extract_method_interval_xcols(x_cols, "sma")
    params = {
        "finger_cols": finger_columns,
        "x_cols": x_cols,
        "exg_fs": exg_fs,
        "ema_methods": ema_methods, "ema_intervals": ema_intervals,
        "sma_methods": sma_methods, "sma_intervals": sma_intervals,
        "mean": train_mean.tolist(), "std": train_std.tolist(),
        "iir_params": iir_params,
        "seq_len": 50, "delay": 4,
        "model": {"type": "SpikingTemporalNet", "hidden": 128, "beta": 0.9, "threshold": 1.0}
    }
    with open(os.path.join(model_dir, "model_params.json"), "w") as f:
        json.dump(params, f, indent=2)

save_model()
print("Saved model + params.")

NameError: name 'model' is not defined

In [9]:
# %% [markdown]
# ## 8) Quick Validation Preview (first batch probabilities)
with torch.no_grad():
    xb,yb = next(iter(val_dl))
    logits = model(xb.to(DEVICE)).cpu().numpy()
    probs  = 1/(1+np.exp(-logits))
print("Preview probs shape (B,T,D):", probs.shape)

NameError: name 'val_dl' is not defined